# Validate Gold facts and dimensions

## Purpose

Check the Gold star schema before any number reaches the dashboard or Genie. This notebook covers issues #60 to #64:

- Issue #60: project fact checks, such as one current row per project and keys that find their dimension rows
- Issue #61: population checks, such as one primary source per region and year
- Issue #62: flood checks, such as levels, severity ranks and the area CRS
- Issue #63: grain and key checks for every fact and dimension
- Issue #64: Gold totals by region compared with Silver

The notebook only reads Gold and Silver. It never changes them. Evidence is saved to `04-validation.gold_dq_results` (D-47) **before** the STOP gate runs, so a failed check stays visible.

## Cost

Each Gold table is read a few times at most. The project fact is the largest, with one row per project and snapshot. All checks come from a few metric rows, and the gate reads the saved results.

## 1. Set the expected values and read the Gold runs

**What:** set the expected values once, then read the run and snapshot behind each Gold fact and dimension, and the Silver runs they came from.

**Why:** the expected values live here, not in the Gold notebooks, so Gold never checks itself. The validation run ID comes from the Gold runs, so a rerun on the same Gold data updates the same evidence instead of adding copies.

**Reference value:** 112,727,776 people is the 2024 census total named in issue #61. It is used only for a FLAG and never changes data.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE validation_rule_version STRING DEFAULT 'gold-validation-v2';
-- the expected values are set here on purpose, so no Gold notebook checks itself
DECLARE OR REPLACE VARIABLE region_key_match_statuses ARRAY<STRING>
    DEFAULT ARRAY('MATCHED', 'MATCHED_WITH_CONFLICT');
DECLARE OR REPLACE VARIABLE flood_key_match_status STRING DEFAULT 'MATCHED';
DECLARE OR REPLACE VARIABLE non_geographic_match_status STRING DEFAULT 'NON_GEOGRAPHIC';
DECLARE OR REPLACE VARIABLE non_geographic_method STRING DEFAULT 'NON_GEOGRAPHIC_RULE';
DECLARE OR REPLACE VARIABLE expected_flood_levels ARRAY<STRING>
    DEFAULT ARRAY('Unknown', 'Low', 'Moderate', 'High', 'Very High');
DECLARE OR REPLACE VARIABLE expected_area_crs STRING DEFAULT 'EPSG:6933';
-- the 2024 census total from earlier evidence, used only for a FLAG
DECLARE OR REPLACE VARIABLE reference_national_population BIGINT DEFAULT 112727776;

CREATE OR REPLACE TEMPORARY VIEW gold_validation_context AS
WITH project_fact AS (
    SELECT
        MAX(run_id) AS project_fact_run_id,
        COUNT(DISTINCT run_id) AS project_fact_run_count,
        MAX(source_snapshot_id) AS project_snapshot_id,
        MAX(delivery_rule_version) AS delivery_rule_version
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot
),
population_fact AS (
    SELECT
        MAX(run_id) AS population_fact_run_id,
        COUNT(DISTINCT run_id) AS population_fact_run_count
    FROM `03-gold`.fact_region_population
),
exposure_fact AS (
    SELECT
        MAX(run_id) AS exposure_fact_run_id,
        COUNT(DISTINCT run_id) AS exposure_fact_run_count,
        MAX(mgb_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(area_rule_version) AS area_rule_version
    FROM `03-gold`.fact_region_flood_exposure
),
dimension_runs AS (
    SELECT
        (SELECT MAX(run_id) FROM `03-gold`.dim_region) AS dim_region_run_id,
        (SELECT MAX(run_id) FROM `03-gold`.dim_region_boundary) AS dim_region_boundary_run_id,
        (SELECT MAX(run_id) FROM `03-gold`.dim_flood_susceptibility) AS dim_flood_run_id,
        (SELECT MAX(run_id) FROM `03-gold`.dim_project_status) AS dim_status_run_id,
        (SELECT MAX(run_id) FROM `03-gold`.dim_project) AS dim_project_run_id
),
silver_runs AS (
    SELECT
        (SELECT MAX(run_id) FROM `02-silver`.silver_project) AS silver_project_run_id,
        (SELECT MAX(source_snapshot_id) FROM `02-silver`.silver_project) AS silver_project_snapshot_id,
        (SELECT MAX(run_id) FROM `02-silver`.silver_project_region_map) AS silver_region_map_run_id,
        (SELECT MAX(run_id) FROM `02-silver`.silver_project_flood_map) AS silver_flood_map_run_id,
        (SELECT MAX(run_id) FROM `02-silver`.silver_region_population) AS silver_population_run_id,
        (SELECT MAX(run_id) FROM `02-silver`.silver_region_flood_exposure) AS silver_exposure_run_id
)
SELECT
    project_fact.*,
    population_fact.*,
    exposure_fact.*,
    dimension_runs.*,
    silver_runs.*,
    -- the same Gold runs always give the same ID, so a rerun updates its evidence
    SHA2(CONCAT_WS(
        '|',
        project_fact.project_fact_run_id,
        population_fact.population_fact_run_id,
        exposure_fact.exposure_fact_run_id,
        validation_rule_version
    ), 256) AS validation_run_id
FROM project_fact
CROSS JOIN population_fact
CROSS JOIN exposure_fact
CROSS JOIN dimension_runs
CROSS JOIN silver_runs;

SELECT * FROM gold_validation_context;

## 2. Project fact metrics (#60, #63)

**What:** join every project fact row to its dimension rows and its Silver project, then count the rows that break a rule.

**Why:** the project fact feeds the main dashboard numbers. A duplicate current row doubles a total. A key without a dimension row drops a project from a chart. Key 0 must appear only in the cases the data model allows. Current rows must come from the DPWH snapshot that Silver holds now.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_validation_project_rows AS
-- each dimension is reduced to one row per key, so a join never repeats a fact row
WITH project_keys AS (
    SELECT
        project_key,
        MAX(contract_id) AS contract_id
    FROM `03-gold`.dim_project
    GROUP BY ALL
),
region_keys AS (
    SELECT DISTINCT region_key FROM `03-gold`.dim_region
),
status_keys AS (
    SELECT DISTINCT status_key FROM `03-gold`.dim_project_status
),
flood_keys AS (
    SELECT DISTINCT flood_susceptibility_key FROM `03-gold`.dim_flood_susceptibility
),
date_keys AS (
    SELECT DISTINCT date_key FROM `03-gold`.dim_date
)
SELECT
    fact.*,
    project.contract_id IS NOT NULL AS in_silver,
    project.source_snapshot_id AS silver_snapshot_id,
    project.status_raw AS silver_status_raw,
    project.start_date AS silver_start_date,
    project.completion_date AS silver_completion_date,
    dim_project.project_key IS NOT NULL AS project_key_resolves,
    region.region_key IS NOT NULL AS region_key_resolves,
    status.status_key IS NOT NULL AS status_key_resolves,
    flood.flood_susceptibility_key IS NOT NULL AS flood_key_resolves,
    snapshot_date.date_key IS NOT NULL
        AND start_date.date_key IS NOT NULL
        AND completion_date.date_key IS NOT NULL AS date_keys_resolve
FROM `03-gold`.fact_project_snapshot AS fact
LEFT JOIN project_keys AS dim_project
    ON fact.project_key = dim_project.project_key
LEFT JOIN region_keys AS region
    ON fact.region_key = region.region_key
LEFT JOIN status_keys AS status
    ON fact.status_key = status.status_key
LEFT JOIN flood_keys AS flood
    ON fact.flood_susceptibility_key = flood.flood_susceptibility_key
LEFT JOIN date_keys AS snapshot_date
    ON fact.snapshot_date_key = snapshot_date.date_key
LEFT JOIN date_keys AS start_date
    ON fact.start_date_key = start_date.date_key
LEFT JOIN date_keys AS completion_date
    ON fact.source_completion_date_key = completion_date.date_key
-- dim_project stores the source system as DPWH, so Silver is matched by Contract ID
LEFT JOIN `02-silver`.silver_project AS project
    ON dim_project.contract_id = project.contract_id;

CREATE OR REPLACE TEMPORARY VIEW gold_validation_project_metrics AS
WITH metrics AS (
    SELECT
        COUNT(*) AS fact_rows,
        COUNT_IF(is_current_snapshot) AS current_rows,
        COUNT_IF(is_current_snapshot)
            - COUNT(DISTINCT CASE WHEN is_current_snapshot THEN project_key END) AS extra_current_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'project', project_key, 'snapshot', source_snapshot_id
        )) AS duplicate_grain_rows,
        COUNT(*) - COUNT(DISTINCT project_snapshot_key) AS duplicate_key_rows,
        COUNT_IF(physical_progress_pct < 0 OR physical_progress_pct > 100) AS progress_out_of_range_rows,
        COUNT_IF(physical_progress_pct IS NULL) AS progress_null_rows,
        COUNT_IF(reported_budget_pesos < 0) AS negative_budget_rows,
        COUNT_IF(reported_budget_pesos IS NULL) AS budget_null_rows,
        COUNT_IF(source_snapshot_id IS NULL OR TRIM(source_snapshot_id) = '') AS missing_snapshot_rows,
        COUNT_IF(NOT project_key_resolves) AS unresolved_project_rows,
        COUNT_IF(NOT region_key_resolves) AS unresolved_region_rows,
        COUNT_IF(NOT status_key_resolves) AS unresolved_status_rows,
        COUNT_IF(NOT flood_key_resolves) AS unresolved_flood_rows,
        COUNT_IF(NOT date_keys_resolve) AS unresolved_date_rows,
        COUNT_IF(NOT (
            project_key_resolves AND region_key_resolves AND status_key_resolves
            AND flood_key_resolves AND date_keys_resolve
        )) AS unresolved_key_rows,
        -- key 0 must appear exactly when the documented case applies, never more and never less
        COUNT_IF((region_key = 0) = COALESCE(
            ARRAY_CONTAINS(region_key_match_statuses, region_match_status), FALSE
        )) AS region_key_rule_breaks,
        COUNT_IF((flood_susceptibility_key = 0) = COALESCE(
            flood_match_status = flood_key_match_status, FALSE
        )) AS flood_key_rule_breaks,
        COUNT_IF(is_current_snapshot AND in_silver
            AND (status_key = 0) <> (silver_status_raw IS NULL)) AS status_key_rule_breaks,
        COUNT_IF(snapshot_date_key = 0 OR (is_current_snapshot AND in_silver AND (
            (start_date_key = 0) <> (silver_start_date IS NULL)
            OR (source_completion_date_key = 0) <> (silver_completion_date IS NULL)
        ))) AS date_key_rule_breaks,
        COUNT_IF(region_match_method = non_geographic_method) AS central_office_rows,
        COUNT_IF(region_match_method = non_geographic_method AND NOT (
            region_key = 0 AND region_match_status <=> non_geographic_match_status
        )) AS central_office_breaks,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR source_snapshot_id IS NULL
            OR delivery_rule_version IS NULL) AS missing_lineage_rows,
        COUNT_IF(is_current_snapshot AND NOT in_silver) AS stale_current_rows,
        -- current rows must come from the DPWH snapshot that Silver holds now
        COUNT_IF(is_current_snapshot AND in_silver
            AND NOT (source_snapshot_id <=> silver_snapshot_id)) AS snapshot_mismatch_rows,
        SUM(CASE WHEN is_current_snapshot THEN reported_budget_pesos END) AS gold_budget_pesos,
        COUNT_IF(is_current_snapshot AND region_key = 0) AS region_key_zero_rows,
        COUNT_IF(is_current_snapshot AND flood_susceptibility_key = 0) AS flood_key_zero_rows,
        COUNT_IF(is_current_snapshot AND status_key = 0) AS status_key_zero_rows,
        COUNT_IF(is_current_snapshot AND long_running_flag IS NULL) AS long_running_null_rows
    FROM gold_validation_project_rows
),
silver AS (
    SELECT
        COUNT(*) AS silver_projects,
        SUM(reported_budget_pesos) AS silver_budget_pesos
    FROM `02-silver`.silver_project
),
flood_input AS (
    -- Sam's note on #60: COUNT(DISTINCT mgb_source_load_ts) = 1 ignores NULL load times
    SELECT
        COUNT(*) AS flood_map_rows,
        COUNT_IF(mgb_source_load_ts IS NULL) AS flood_map_missing_load_ts_rows
    FROM `02-silver`.silver_project_flood_map
),
key_zero_buckets AS (
    SELECT
        'region' AS kind,
        COALESCE(region_match_status, 'NULL') AS bucket,
        COUNT(*) AS bucket_rows
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot AND region_key = 0
    GROUP BY ALL
    UNION ALL
    SELECT
        'flood' AS kind,
        CONCAT(COALESCE(flood_match_status, 'NULL'), ' with ', COALESCE(coordinate_status, 'NULL')) AS bucket,
        COUNT(*) AS bucket_rows
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot AND flood_susceptibility_key = 0
    GROUP BY ALL
),
breakdown AS (
    SELECT
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(
            CASE WHEN kind = 'region' THEN CONCAT(bucket, '=', bucket_rows) END
        ))) AS region_key_zero_by_status,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_LIST(
            CASE WHEN kind = 'flood' THEN CONCAT(bucket, '=', bucket_rows) END
        ))) AS flood_key_zero_by_status
    FROM key_zero_buckets
)
SELECT
    metrics.*,
    silver.*,
    flood_input.*,
    breakdown.*
FROM metrics
CROSS JOIN silver
CROSS JOIN flood_input
CROSS JOIN breakdown;

SELECT * FROM gold_validation_project_metrics;

## 3. Gold and Silver by region (#64)

**What:** count projects and add up the reported budget for each region, in Silver and in Gold.

**Why:** the totals must match region by region, not only overall. Gold gives a real region key only to `MATCHED` and `MATCHED_WITH_CONFLICT` projects. The Silver side uses the same rule, so every other project goes in the key 0 bucket on both sides.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_validation_region_reconciliation AS
WITH silver AS (
    -- the Silver side uses the same rule as Gold: only these statuses get a real region key
    SELECT
        CASE
            WHEN ARRAY_CONTAINS(region_key_match_statuses, region_map.match_status)
                THEN region_map.psgc_region_code
            ELSE 'KEY_0'
        END AS region_code,
        COUNT(*) AS silver_projects,
        SUM(project.reported_budget_pesos) AS silver_budget_pesos
    FROM `02-silver`.silver_project AS project
    INNER JOIN `02-silver`.silver_project_region_map AS region_map
        ON project.project_key = region_map.project_key
    GROUP BY ALL
),
region_codes AS (
    SELECT
        region_key,
        MAX(psgc_region_code) AS psgc_region_code
    FROM `03-gold`.dim_region
    GROUP BY ALL
),
gold AS (
    SELECT
        CASE WHEN fact.region_key = 0 THEN 'KEY_0' ELSE region.psgc_region_code END AS region_code,
        COUNT(*) AS gold_projects,
        SUM(fact.reported_budget_pesos) AS gold_budget_pesos
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN region_codes AS region
        ON fact.region_key = region.region_key
    WHERE fact.is_current_snapshot
    GROUP BY ALL
)
SELECT
    COALESCE(silver.region_code, gold.region_code, 'NULL') AS region_code,
    COALESCE(silver.silver_projects, 0) AS silver_projects,
    COALESCE(gold.gold_projects, 0) AS gold_projects,
    silver.silver_budget_pesos,
    gold.gold_budget_pesos,
    COALESCE(silver.silver_projects, 0) = COALESCE(gold.gold_projects, 0) AS projects_match,
    silver.silver_budget_pesos <=> gold.gold_budget_pesos AS budget_match
FROM silver
FULL OUTER JOIN gold
    ON silver.region_code <=> gold.region_code;

SELECT * FROM gold_validation_region_reconciliation ORDER BY region_code;

## 4. Population and flood exposure metrics (#61, #62, #63)

**What:** check the grain, keys, values and totals of the two regional facts.

**Why:** per-person numbers answer our main question, so each official region needs exactly one primary population row. Key 0 never enters a population ratio. Flood areas must be in square kilometres from the equal-area CRS, and their totals must match Silver. These two facts have no current flag, and their `MERGE` never deletes. So Gold must have exactly the Silver rows, or old rows from an earlier version would double a total.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_validation_population_metrics AS
WITH region_keys AS (
    SELECT DISTINCT region_key FROM `03-gold`.dim_region
),
official_regions AS (
    SELECT DISTINCT region_key
    FROM `03-gold`.dim_region
    WHERE region_key <> 0 AND is_geographic_region
),
fact AS (
    SELECT
        population.*,
        region.region_key IS NOT NULL AS region_key_resolves
    FROM `03-gold`.fact_region_population AS population
    LEFT JOIN region_keys AS region
        ON population.region_key = region.region_key
),
fact_metrics AS (
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'region', region_key, 'year', reference_year, 'source', source_name
        )) AS duplicate_grain_rows,
        COUNT(*) - COUNT(DISTINCT region_population_fact_key) AS duplicate_key_rows,
        -- an empty count is not above zero, so it fails too
        COUNT_IF(is_primary_source AND (population_count IS NULL OR population_count <= 0)) AS non_positive_rows,
        COUNT_IF(region_key IS NULL OR region_key = 0) AS key_zero_rows,
        COUNT_IF(NOT region_key_resolves) AS unresolved_region_rows,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR source_name IS NULL
            OR reference_year IS NULL) AS missing_lineage_rows,
        COUNT_IF(is_primary_source) AS primary_rows,
        SUM(CASE WHEN is_primary_source THEN population_count END) AS primary_people,
        SUM(population_count) AS gold_people
    FROM fact
),
fact_years AS (
    SELECT DISTINCT reference_year FROM fact
),
region_year_primary AS (
    -- every official region needs exactly one primary row in each year the fact holds
    SELECT
        region.region_key,
        years.reference_year,
        COUNT(fact.region_key) AS primary_rows
    FROM official_regions AS region
    CROSS JOIN fact_years AS years
    LEFT JOIN fact
        ON region.region_key = fact.region_key
        AND years.reference_year = fact.reference_year
        AND fact.is_primary_source
    GROUP BY ALL
),
one_primary AS (
    SELECT
        COUNT(*) AS region_years,
        COUNT_IF(primary_rows <> 1) AS region_years_without_one_primary
    FROM region_year_primary
),
silver AS (
    SELECT
        COUNT(*) AS silver_rows,
        SUM(population_count) AS silver_people
    FROM `02-silver`.silver_region_population
),
table_c AS (
    -- unmatched Table C rows inside a known town explain the gap to the national census total
    SELECT
        COUNT(*) AS table_c_rows,
        COUNT_IF(row_disposition IN ('UNMATCHED', 'AMBIGUOUS') AND locality_context_code IS NOT NULL) AS unplaced_rows,
        COALESCE(SUM(CASE
            WHEN row_disposition IN ('UNMATCHED', 'AMBIGUOUS') AND locality_context_code IS NOT NULL THEN population_count
        END), 0) AS unplaced_people,
        CONCAT_WS(', ', SLICE(SORT_ARRAY(COLLECT_LIST(CASE
            WHEN row_disposition IN ('UNMATCHED', 'AMBIGUOUS') AND locality_context_code IS NOT NULL THEN CONCAT(
                place_name_raw, ' in sheet ', sheet_name, ', region code ',
                COALESCE(CONCAT(LEFT(locality_context_code, 2), '00000000'), 'unknown'),
                ', ', COALESCE(CAST(population_count AS STRING), 'no count'), ' people, ',
                row_disposition
            )
        END)), 1, 10)) AS unplaced_sample
    FROM `02-silver`.silver_population_place_reconciliation
)
SELECT
    fact_metrics.*,
    one_primary.*,
    silver.*,
    table_c.*
FROM fact_metrics
CROSS JOIN one_primary
CROSS JOIN silver
CROSS JOIN table_c;

CREATE OR REPLACE TEMPORARY VIEW gold_validation_exposure_metrics AS
WITH region_keys AS (
    SELECT DISTINCT region_key FROM `03-gold`.dim_region
),
flood_keys AS (
    SELECT DISTINCT flood_susceptibility_key FROM `03-gold`.dim_flood_susceptibility
),
fact AS (
    SELECT
        exposure.*,
        region.region_key IS NOT NULL AS region_key_resolves,
        flood.flood_susceptibility_key IS NOT NULL AS flood_key_resolves
    FROM `03-gold`.fact_region_flood_exposure AS exposure
    LEFT JOIN region_keys AS region
        ON exposure.region_key = region.region_key
    LEFT JOIN flood_keys AS flood
        ON exposure.flood_susceptibility_key = flood.flood_susceptibility_key
),
fact_metrics AS (
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'region', region_key, 'level', flood_susceptibility_key,
            'mgb', mgb_source_snapshot_id, 'boundary', boundary_version
        )) AS duplicate_grain_rows,
        COUNT(*) - COUNT(DISTINCT region_flood_exposure_key) AS duplicate_key_rows,
        COUNT_IF(susceptible_area_sqkm < 0) AS negative_area_rows,
        COUNT_IF(share_of_region_area_pct < 0 OR share_of_region_area_pct > 100) AS share_out_of_range_rows,
        COUNT_IF(area_calculation_crs IS NULL OR area_calculation_crs <> expected_area_crs) AS wrong_crs_rows,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(COALESCE(area_calculation_crs, 'NULL')))) AS crs_found,
        COUNT_IF(region_key IS NULL OR region_key = 0
            OR flood_susceptibility_key IS NULL OR flood_susceptibility_key = 0) AS key_zero_rows,
        COUNT_IF(NOT (region_key_resolves AND flood_key_resolves)) AS unresolved_key_rows,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR mgb_source_snapshot_id IS NULL
            OR boundary_version IS NULL OR area_rule_version IS NULL) AS missing_lineage_rows,
        COUNT_IF(susceptible_area_sqkm IS NULL) AS null_area_rows,
        SUM(susceptible_area_sqkm) AS gold_area_sqkm
    FROM fact
),
silver AS (
    SELECT
        COUNT(*) AS silver_rows,
        SUM(susceptible_area_sqkm) AS silver_area_sqkm,
        COUNT_IF(susceptible_area_sqkm IS NULL) AS silver_null_area_rows
    FROM `02-silver`.silver_region_flood_exposure
)
SELECT
    fact_metrics.*,
    silver.*
FROM fact_metrics
CROSS JOIN silver;

SELECT * FROM gold_validation_population_metrics;
SELECT * FROM gold_validation_exposure_metrics;

## 5. Dimension metrics (#62, #63)

**What:** check the uniqueness rule, the key 0 row and the lineage columns of each dimension. Check that the flood levels and their severity ranks follow the MGB rules.

**Why:** a duplicated dimension key repeats fact rows in every join. A missing key 0 row breaks the facts that need it. `dim_date` is built from a date range with no source, so it has no lineage columns to check.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_validation_dimension_metrics AS
WITH date_metrics AS (
    SELECT
        COUNT(*) AS date_rows,
        COUNT(*) - COUNT(DISTINCT date_key) AS date_duplicate_key_rows,
        COUNT_IF(calendar_date IS NOT NULL) - COUNT(DISTINCT calendar_date) AS date_duplicate_grain_rows,
        COUNT_IF(date_key = 0) AS date_key_zero_rows,
        COUNT_IF(date_key <> 0 AND (calendar_date IS NULL
            OR date_key <> CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT))) AS date_key_format_breaks
    FROM `03-gold`.dim_date
),
region_metrics AS (
    SELECT
        COUNT(*) AS region_rows,
        COUNT(*) - COUNT(DISTINCT region_key) AS region_duplicate_key_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'code', psgc_region_code, 'version', psgc_version
        )) AS region_duplicate_grain_rows,
        COUNT_IF(region_key = 0) AS region_key_zero_rows,
        COUNT_IF(region_key = 0 AND NOT (is_geographic_region <=> FALSE)) AS region_key_zero_geographic_rows,
        COUNT_IF(region_key <> 0) AS region_official_rows,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR psgc_run_id IS NULL
            OR exposure_run_id IS NULL) AS region_missing_lineage_rows
    FROM `03-gold`.dim_region
),
official_region_keys AS (
    SELECT DISTINCT region_key FROM `03-gold`.dim_region WHERE region_key <> 0
),
boundary_metrics AS (
    SELECT
        COUNT(*) AS boundary_rows,
        COUNT(*) - COUNT(DISTINCT boundary.boundary_key) AS boundary_duplicate_key_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'region', boundary.region_key, 'version', boundary.boundary_version
        )) AS boundary_duplicate_grain_rows,
        COUNT_IF(boundary.region_key IS NULL OR boundary.region_key = 0
            OR region.region_key IS NULL) AS boundary_unresolved_region_rows,
        COUNT_IF(NOT COALESCE(boundary.geometry_valid_flag, FALSE)) AS boundary_invalid_geometry_rows,
        COUNT_IF(boundary.run_id IS NULL OR boundary.source_load_ts IS NULL
            OR boundary.boundary_version IS NULL OR boundary.psgc_run_id IS NULL
            OR boundary.exposure_run_id IS NULL) AS boundary_missing_lineage_rows
    FROM `03-gold`.dim_region_boundary AS boundary
    LEFT JOIN official_region_keys AS region
        ON boundary.region_key = region.region_key
),
flood_metrics AS (
    SELECT
        COUNT(*) AS flood_rows,
        COUNT(*) - COUNT(DISTINCT flood_susceptibility_key) AS flood_duplicate_key_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'system', source_system, 'level', flood_susceptibility_level, 'version', source_version
        )) AS flood_duplicate_grain_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'system', source_system, 'version', source_version, 'rank', severity_rank
        )) AS flood_duplicate_rank_rows,
        COUNT_IF(NOT ARRAY_CONTAINS(expected_flood_levels, flood_susceptibility_level)
            OR flood_susceptibility_level IS NULL) AS flood_unexpected_level_rows,
        SIZE(ARRAY_EXCEPT(
            expected_flood_levels, COLLECT_SET(flood_susceptibility_level)
        )) AS flood_missing_levels,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CONCAT(
            flood_susceptibility_level, '=', CAST(severity_rank AS STRING)
        )))) AS flood_levels_found,
        COUNT_IF(flood_susceptibility_key = 0) AS flood_key_zero_rows,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR source_version IS NULL)
            AS flood_missing_lineage_rows
    FROM `03-gold`.dim_flood_susceptibility
),
status_metrics AS (
    SELECT
        COUNT(*) AS status_rows,
        COUNT(*) - COUNT(DISTINCT status_key) AS status_duplicate_key_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'system', source_system, 'status', source_status, 'version', status_mapping_version
        )) AS status_duplicate_grain_rows,
        COUNT_IF(status_key = 0) AS status_key_zero_rows,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR status_mapping_version IS NULL)
            AS status_missing_lineage_rows,
        COUNT_IF(is_active) AS status_active_rows,
        -- active statuses that are really numbers or map points, from DPWH rows split by a line break
        COUNT_IF(is_active AND status_key <> 0
            AND source_status RLIKE '^[-0-9.,° ()NSEW]*[0-9][-0-9.,° ()NSEW]*$') AS status_shifted_value_rows,
        CONCAT_WS(', ', SLICE(SORT_ARRAY(COLLECT_LIST(CASE
            WHEN is_active AND status_key <> 0
                AND source_status RLIKE '^[-0-9.,° ()NSEW]*[0-9][-0-9.,° ()NSEW]*$'
            THEN source_status
        END)), 1, 10)) AS status_shifted_value_sample
    FROM `03-gold`.dim_project_status
),
project_metrics AS (
    SELECT
        COUNT(*) AS project_rows,
        COUNT(*) - COUNT(DISTINCT project_key) AS project_duplicate_key_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'system', source_system, 'contract', contract_id
        )) AS project_duplicate_grain_rows,
        COUNT_IF(run_id IS NULL OR source_load_ts IS NULL OR taxonomy_version IS NULL)
            AS project_missing_lineage_rows
    FROM `03-gold`.dim_project
),
silver_projects AS (
    SELECT COUNT(*) AS project_silver_rows
    FROM `02-silver`.silver_project
),
silver_regions AS (
    SELECT COUNT(*) AS silver_region_rows
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
)
SELECT
    date_metrics.*,
    region_metrics.*,
    boundary_metrics.*,
    flood_metrics.*,
    status_metrics.*,
    project_metrics.*,
    silver_projects.*,
    silver_regions.*
FROM date_metrics
CROSS JOIN region_metrics
CROSS JOIN boundary_metrics
CROSS JOIN flood_metrics
CROSS JOIN status_metrics
CROSS JOIN project_metrics
CROSS JOIN silver_projects
CROSS JOIN silver_regions;

SELECT * FROM gold_validation_dimension_metrics;

## 6. Turn the metrics into checks

**STOP** checks protect grain, keys, valid values, totals, lineage and the current snapshot. A failed STOP check is `FAIL`.

**FLAG** checks keep known data limits visible with their counts, such as projects without a region or a flood level. A FLAG is not a failure. Zero findings are `PASS`.

Every check names the columns it tested in `details`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_validation_reconciliation_summary AS
SELECT
    COUNT(*) AS region_buckets,
    COUNT_IF(NOT projects_match) AS project_count_breaks,
    COUNT_IF(NOT budget_match) AS budget_breaks,
    CONCAT_WS(', ', COLLECT_LIST(CASE
        WHEN NOT projects_match OR NOT budget_match THEN CONCAT(
            region_code, ' Silver ', silver_projects, ' projects and ',
            COALESCE(CAST(silver_budget_pesos AS STRING), 'NULL'), ' pesos, Gold ', gold_projects,
            ' projects and ', COALESCE(CAST(gold_budget_pesos AS STRING), 'NULL'), ' pesos'
        )
    END)) AS differences
FROM gold_validation_region_reconciliation;

CREATE OR REPLACE TEMPORARY VIEW gold_validation_checks AS
WITH checks AS (
    -- each metrics row is turned into check rows, so no table is scanned again per check
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Uniqueness',
            'check_name', 'At most one current row per project',
            'failed_rows', CAST(project.extra_current_rows AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_key, is_current_snapshot. Current rows=', FORMAT_NUMBER(project.current_rows, 0))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Uniqueness',
            'check_name', 'One row per project and snapshot',
            'failed_rows', CAST(project.duplicate_grain_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key, source_snapshot_id'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Uniqueness',
            'check_name', 'Project snapshot key is unique',
            'failed_rows', CAST(project.duplicate_key_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_snapshot_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Validity',
            'check_name', 'Physical progress is 0 to 100 or empty',
            'failed_rows', CAST(project.progress_out_of_range_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: physical_progress_pct. Empty values=', FORMAT_NUMBER(project.progress_null_rows, 0))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Validity',
            'check_name', 'Reported budget is zero or more, or empty',
            'failed_rows', CAST(project.negative_budget_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: reported_budget_pesos. Empty values=', FORMAT_NUMBER(project.budget_null_rows, 0))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Source snapshot ID is never empty',
            'failed_rows', CAST(project.missing_snapshot_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_snapshot_id'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Consistency',
            'check_name', 'Every key finds its dimension row',
            'failed_rows', CAST(project.unresolved_key_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: project_key, region_key, status_key, flood_susceptibility_key, snapshot_date_key, start_date_key, source_completion_date_key. Missing project=', project.unresolved_project_rows, ', region=', project.unresolved_region_rows, ', status=', project.unresolved_status_rows, ', flood=', project.unresolved_flood_rows, ', date=', project.unresolved_date_rows)
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Consistency',
            'check_name', 'Region key 0 only when the region match is not MATCHED or MATCHED_WITH_CONFLICT',
            'failed_rows', CAST(project.region_key_rule_breaks AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key, region_match_status. Key 0 rows by status: ', COALESCE(project.region_key_zero_by_status, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Consistency',
            'check_name', 'Flood key 0 only when the flood match is not MATCHED',
            'failed_rows', CAST(project.flood_key_rule_breaks AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: flood_susceptibility_key, flood_match_status. Key 0 rows by status and coordinates: ', COALESCE(project.flood_key_zero_by_status, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Consistency',
            'check_name', 'Status key 0 only when the Silver status is missing or conflicting',
            'failed_rows', CAST(project.status_key_rule_breaks AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: status_key, silver_project.status_raw. Silver leaves status_raw empty when the status is missing or its rows disagree'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Consistency',
            'check_name', 'Date key 0 only when the date is missing',
            'failed_rows', CAST(project.date_key_rule_breaks AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: snapshot_date_key, start_date_key, source_completion_date_key, silver_project.start_date, silver_project.completion_date. The snapshot date key is never 0'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Consistency',
            'check_name', 'Central Office projects use region key 0 and the non-geographic status',
            'failed_rows', CAST(project.central_office_breaks AS BIGINT), 'total_rows', CAST(project.central_office_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key, region_match_status, region_match_method. Central Office rows=', FORMAT_NUMBER(project.central_office_rows, 0))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(project.missing_lineage_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, source_snapshot_id, delivery_rule_version'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Current rows equal Silver projects, with no stale current rows',
            'failed_rows', CAST(ABS(project.current_rows - project.silver_projects) + project.stale_current_rows AS BIGINT), 'total_rows', CAST(project.silver_projects AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: is_current_snapshot, project_key. Gold current=', FORMAT_NUMBER(project.current_rows, 0), ', Silver=', FORMAT_NUMBER(project.silver_projects, 0), ', stale current rows=', FORMAT_NUMBER(project.stale_current_rows, 0))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Timeliness',
            'check_name', 'Current rows use the Silver DPWH snapshot',
            'failed_rows', CAST(project.snapshot_mismatch_rows AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: source_snapshot_id, silver_project.source_snapshot_id. Gold snapshot=', COALESCE(context.project_snapshot_id, 'NULL'), ', Silver snapshot=', COALESCE(context.silver_project_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Accuracy',
            'check_name', 'Current budget total equals Silver',
            'failed_rows', CAST(CASE WHEN project.gold_budget_pesos <=> project.silver_budget_pesos THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: reported_budget_pesos. Gold=', COALESCE(CAST(project.gold_budget_pesos AS STRING), 'NULL'), ', Silver=', COALESCE(CAST(project.silver_budget_pesos AS STRING), 'NULL'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Accuracy',
            'check_name', 'Project count by region equals Silver',
            'failed_rows', CAST(reconciliation.project_count_breaks AS BIGINT), 'total_rows', CAST(reconciliation.region_buckets AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key, project_key. Differences: ', COALESCE(NULLIF(reconciliation.differences, ''), 'none'), '. Gold run=', context.project_fact_run_id, ', Silver project run=', context.silver_project_run_id, ', Silver region map run=', context.silver_region_map_run_id, ', snapshot=', context.project_snapshot_id)
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Accuracy',
            'check_name', 'Budget by region equals Silver',
            'failed_rows', CAST(reconciliation.budget_breaks AS BIGINT), 'total_rows', CAST(reconciliation.region_buckets AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key, reported_budget_pesos. Differences: ', COALESCE(NULLIF(reconciliation.differences, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Projects without a region (key 0)',
            'failed_rows', CAST(project.region_key_zero_rows AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: region_key, region_match_status. By status: ', COALESCE(project.region_key_zero_by_status, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Projects without a flood level (key 0)',
            'failed_rows', CAST(project.flood_key_zero_rows AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: flood_susceptibility_key, flood_match_status, coordinate_status. By status: ', COALESCE(project.flood_key_zero_by_status, 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Projects without a status (key 0)',
            'failed_rows', CAST(project.status_key_zero_rows AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: status_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Progress not published because Silver did not mark it VALID',
            'failed_rows', CAST(project.progress_null_rows AS BIGINT), 'total_rows', CAST(project.fact_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: physical_progress_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_project_snapshot', 'attribute', 'Completeness',
            'check_name', 'Long-running flag is empty: no start date, or completion before start',
            'failed_rows', CAST(project.long_running_null_rows AS BIGINT), 'total_rows', CAST(project.current_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: long_running_flag, start_date_key, source_completion_date_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Uniqueness',
            'check_name', 'One row per region, year and source',
            'failed_rows', CAST(population.duplicate_grain_rows AS BIGINT), 'total_rows', CAST(population.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, reference_year, source_name'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Uniqueness',
            'check_name', 'Population fact key is unique',
            'failed_rows', CAST(population.duplicate_key_rows AS BIGINT), 'total_rows', CAST(population.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_population_fact_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Validity',
            'check_name', 'Primary population is above zero',
            'failed_rows', CAST(population.non_positive_rows AS BIGINT), 'total_rows', CAST(population.primary_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: population_count, is_primary_source. An empty count fails too'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Uniqueness',
            'check_name', 'Each region has exactly one primary source per year',
            'failed_rows', CAST(population.region_years_without_one_primary AS BIGINT), 'total_rows', CAST(population.region_years AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, reference_year, is_primary_source. Checked for every official region in dim_region'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Consistency',
            'check_name', 'No population row uses region key 0, so ratios leave it out',
            'failed_rows', CAST(population.key_zero_rows AS BIGINT), 'total_rows', CAST(population.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key. Denominator for per-person ratios: ', FORMAT_NUMBER(population.primary_people, 0), ' people in ', population.primary_rows, ' primary rows. Key 0 is never in it')
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Consistency',
            'check_name', 'Every region key finds its dimension row',
            'failed_rows', CAST(population.unresolved_region_rows AS BIGINT), 'total_rows', CAST(population.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Accuracy',
            'check_name', 'Population total equals Silver',
            'failed_rows', CAST(CASE WHEN population.gold_people <=> population.silver_people THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: population_count. Gold=', FORMAT_NUMBER(population.gold_people, 0), ', Silver=', FORMAT_NUMBER(population.silver_people, 0), '. Gold run=', context.population_fact_run_id, ', Silver run=', context.silver_population_run_id)
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Completeness',
            'check_name', 'Rows equal Silver, with no old rows left',
            'failed_rows', CAST(ABS(population.fact_rows - population.silver_rows) AS BIGINT), 'total_rows', CAST(population.silver_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: all rows, COUNT(*). Gold rows=', population.fact_rows, ', Silver rows=', population.silver_rows, '. The MERGE never deletes, so rows from an older PSGC version would show here')
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(population.missing_lineage_rows AS BIGINT), 'total_rows', CAST(population.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, source_name, reference_year'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_population', 'attribute', 'Accuracy',
            'check_name', 'Region total is below the 2024 census total',
            'failed_rows', CAST(population.unplaced_rows AS BIGINT), 'total_rows', CAST(population.table_c_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: population_count, is_primary_source, silver_population_place_reconciliation.row_disposition, locality_context_code. Gold primary total=', FORMAT_NUMBER(population.primary_people, 0), ', census reference=', FORMAT_NUMBER(reference_national_population, 0), ', gap=', FORMAT_NUMBER(reference_national_population - population.primary_people, 0), ' people. Table C rows in a known town that Silver could not match: ', population.unplaced_rows, ' rows, ', FORMAT_NUMBER(population.unplaced_people, 0), ' people: ', COALESCE(NULLIF(population.unplaced_sample, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Uniqueness',
            'check_name', 'One row per region, level, MGB snapshot and boundary version',
            'failed_rows', CAST(exposure.duplicate_grain_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, flood_susceptibility_key, mgb_source_snapshot_id, boundary_version'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Uniqueness',
            'check_name', 'Flood exposure key is unique',
            'failed_rows', CAST(exposure.duplicate_key_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_flood_exposure_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Validity',
            'check_name', 'Flood area is zero or more',
            'failed_rows', CAST(exposure.negative_area_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: susceptible_area_sqkm'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Validity',
            'check_name', 'Share of region area is 0 to 100',
            'failed_rows', CAST(exposure.share_out_of_range_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: share_of_region_area_pct'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Validity',
            'check_name', 'Areas use the equal-area CRS, not degrees',
            'failed_rows', CAST(exposure.wrong_crs_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: area_calculation_crs. Expected ', expected_area_crs, ', found ', exposure.crs_found)
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Consistency',
            'check_name', 'Every row has a real region and flood level',
            'failed_rows', CAST(exposure.key_zero_rows + exposure.unresolved_key_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, flood_susceptibility_key'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Accuracy',
            'check_name', 'Area total and empty areas equal Silver',
            'failed_rows', CAST(CASE WHEN exposure.gold_area_sqkm <=> exposure.silver_area_sqkm AND exposure.null_area_rows = exposure.silver_null_area_rows THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: susceptible_area_sqkm. Gold=', COALESCE(CAST(exposure.gold_area_sqkm AS STRING), 'NULL'), ' sq km, Silver=', COALESCE(CAST(exposure.silver_area_sqkm AS STRING), 'NULL'), ' sq km. Empty areas Gold=', exposure.null_area_rows, ', Silver=', exposure.silver_null_area_rows, '. Gold run=', context.exposure_fact_run_id, ', Silver run=', context.silver_exposure_run_id)
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Completeness',
            'check_name', 'Rows equal Silver, with no old rows left',
            'failed_rows', CAST(ABS(exposure.fact_rows - exposure.silver_rows) AS BIGINT), 'total_rows', CAST(exposure.silver_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: all rows, COUNT(*). Gold rows=', exposure.fact_rows, ', Silver rows=', exposure.silver_rows, '. The MERGE never deletes, so rows from an older MGB snapshot or boundary version would show here')
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(exposure.missing_lineage_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, mgb_source_snapshot_id, boundary_version, area_rule_version. mgb_source_version is empty by design'
        ),
        NAMED_STRUCT(
            'table_name', 'fact_region_flood_exposure', 'attribute', 'Completeness',
            'check_name', 'Region and level pairs with no area data',
            'failed_rows', CAST(exposure.null_area_rows AS BIGINT), 'total_rows', CAST(exposure.fact_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: susceptible_area_sqkm, spatial_match_status'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_date', 'attribute', 'Uniqueness',
            'check_name', 'One row per calendar date',
            'failed_rows', CAST(dims.date_duplicate_grain_rows AS BIGINT), 'total_rows', CAST(dims.date_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: calendar_date'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_date', 'attribute', 'Uniqueness',
            'check_name', 'Date key is unique',
            'failed_rows', CAST(dims.date_duplicate_key_rows AS BIGINT), 'total_rows', CAST(dims.date_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: date_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_date', 'attribute', 'Completeness',
            'check_name', 'Date key 0 exists once for missing dates',
            'failed_rows', CAST(ABS(dims.date_key_zero_rows - 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', 'Columns: date_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_date', 'attribute', 'Validity',
            'check_name', 'Date key is the date as yyyymmdd',
            'failed_rows', CAST(dims.date_key_format_breaks AS BIGINT), 'total_rows', CAST(dims.date_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: date_key, calendar_date'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region', 'attribute', 'Uniqueness',
            'check_name', 'One row per region code and PSGC version',
            'failed_rows', CAST(dims.region_duplicate_grain_rows AS BIGINT), 'total_rows', CAST(dims.region_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: psgc_region_code, psgc_version'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region', 'attribute', 'Uniqueness',
            'check_name', 'Region key is unique',
            'failed_rows', CAST(dims.region_duplicate_key_rows AS BIGINT), 'total_rows', CAST(dims.region_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region', 'attribute', 'Completeness',
            'check_name', 'Region key 0 exists once and is not a geographic region',
            'failed_rows', CAST(ABS(dims.region_key_zero_rows - 1) + dims.region_key_zero_geographic_rows AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, is_geographic_region'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region', 'attribute', 'Completeness',
            'check_name', 'Official regions equal Silver REGION rows',
            'failed_rows', CAST(ABS(dims.region_official_rows - dims.silver_region_rows) AS BIGINT), 'total_rows', CAST(dims.silver_region_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: region_key. Gold=', dims.region_official_rows, ', Silver=', dims.silver_region_rows)
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(dims.region_missing_lineage_rows AS BIGINT), 'total_rows', CAST(dims.region_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, psgc_run_id, exposure_run_id'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region_boundary', 'attribute', 'Uniqueness',
            'check_name', 'One row per region and boundary version',
            'failed_rows', CAST(dims.boundary_duplicate_grain_rows AS BIGINT), 'total_rows', CAST(dims.boundary_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key, boundary_version'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region_boundary', 'attribute', 'Uniqueness',
            'check_name', 'Boundary key is unique',
            'failed_rows', CAST(dims.boundary_duplicate_key_rows AS BIGINT), 'total_rows', CAST(dims.boundary_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: boundary_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region_boundary', 'attribute', 'Consistency',
            'check_name', 'Every boundary belongs to a real region',
            'failed_rows', CAST(dims.boundary_unresolved_region_rows AS BIGINT), 'total_rows', CAST(dims.boundary_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: region_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region_boundary', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(dims.boundary_missing_lineage_rows AS BIGINT), 'total_rows', CAST(dims.boundary_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, boundary_version, psgc_run_id, exposure_run_id'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region_boundary', 'attribute', 'Validity',
            'check_name', 'Boundary shapes marked not valid',
            'failed_rows', CAST(dims.boundary_invalid_geometry_rows AS BIGINT), 'total_rows', CAST(dims.boundary_rows AS BIGINT),
            'action', 'flag',
            'details', 'Columns: geometry_valid_flag'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_region_boundary', 'attribute', 'Completeness',
            'check_name', 'Official regions without a boundary',
            'failed_rows', CAST(GREATEST(dims.region_official_rows - dims.boundary_rows, 0) AS BIGINT), 'total_rows', CAST(dims.region_official_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: region_key. Regions=', dims.region_official_rows, ', boundaries=', dims.boundary_rows)
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Uniqueness',
            'check_name', 'One row per source system, level and version',
            'failed_rows', CAST(dims.flood_duplicate_grain_rows AS BIGINT), 'total_rows', CAST(dims.flood_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_system, flood_susceptibility_level, source_version'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Uniqueness',
            'check_name', 'Flood susceptibility key is unique',
            'failed_rows', CAST(dims.flood_duplicate_key_rows AS BIGINT), 'total_rows', CAST(dims.flood_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_susceptibility_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Uniqueness',
            'check_name', 'Severity rank is unique within each source system and version',
            'failed_rows', CAST(dims.flood_duplicate_rank_rows AS BIGINT), 'total_rows', CAST(dims.flood_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: source_system, source_version, severity_rank. Found ', dims.flood_levels_found)
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Validity',
            'check_name', 'Levels are Unknown, Low, Moderate, High or Very High',
            'failed_rows', CAST(dims.flood_unexpected_level_rows AS BIGINT), 'total_rows', CAST(dims.flood_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: flood_susceptibility_level. Found ', dims.flood_levels_found)
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Completeness',
            'check_name', 'All five levels are present',
            'failed_rows', CAST(dims.flood_missing_levels AS BIGINT), 'total_rows', CAST(5 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: flood_susceptibility_level. Found ', dims.flood_levels_found)
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Completeness',
            'check_name', 'Flood key 0 exists once for Unknown',
            'failed_rows', CAST(ABS(dims.flood_key_zero_rows - 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', 'Columns: flood_susceptibility_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(dims.flood_missing_lineage_rows AS BIGINT), 'total_rows', CAST(dims.flood_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, source_version'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_flood_susceptibility', 'attribute', 'Auditability',
            'check_name', 'Silver project flood map has a load time on every row',
            'failed_rows', CAST(project.flood_map_missing_load_ts_rows AS BIGINT), 'total_rows', CAST(project.flood_map_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: silver_project_flood_map.mgb_source_load_ts. A count of distinct load times skips empty ones'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project_status', 'attribute', 'Uniqueness',
            'check_name', 'One row per source system, status and mapping version',
            'failed_rows', CAST(dims.status_duplicate_grain_rows AS BIGINT), 'total_rows', CAST(dims.status_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_system, source_status, status_mapping_version'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project_status', 'attribute', 'Uniqueness',
            'check_name', 'Status key is unique',
            'failed_rows', CAST(dims.status_duplicate_key_rows AS BIGINT), 'total_rows', CAST(dims.status_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: status_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project_status', 'attribute', 'Completeness',
            'check_name', 'Status key 0 exists once for missing or conflicting status',
            'failed_rows', CAST(ABS(dims.status_key_zero_rows - 1) AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', 'Columns: status_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project_status', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(dims.status_missing_lineage_rows AS BIGINT), 'total_rows', CAST(dims.status_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, status_mapping_version'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project_status', 'attribute', 'Validity',
            'check_name', 'Active statuses that look like numbers or map points',
            'failed_rows', CAST(dims.status_shifted_value_rows AS BIGINT), 'total_rows', CAST(dims.status_active_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Columns: source_status. These come from DPWH rows split by a line break. Values: ', COALESCE(NULLIF(dims.status_shifted_value_sample, ''), 'none'))
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project', 'attribute', 'Uniqueness',
            'check_name', 'One row per source system and Contract ID',
            'failed_rows', CAST(dims.project_duplicate_grain_rows AS BIGINT), 'total_rows', CAST(dims.project_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: source_system, contract_id'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project', 'attribute', 'Uniqueness',
            'check_name', 'Project key is unique',
            'failed_rows', CAST(dims.project_duplicate_key_rows AS BIGINT), 'total_rows', CAST(dims.project_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: project_key'
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project', 'attribute', 'Completeness',
            'check_name', 'Rows equal Silver projects',
            'failed_rows', CAST(ABS(dims.project_rows - dims.project_silver_rows) AS BIGINT), 'total_rows', CAST(dims.project_silver_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Columns: all rows, COUNT(*). Gold rows=', dims.project_rows, ', Silver rows=', dims.project_silver_rows)
        ),
        NAMED_STRUCT(
            'table_name', 'dim_project', 'attribute', 'Auditability',
            'check_name', 'Lineage columns are filled',
            'failed_rows', CAST(dims.project_missing_lineage_rows AS BIGINT), 'total_rows', CAST(dims.project_rows AS BIGINT),
            'action', 'stop',
            'details', 'Columns: run_id, source_load_ts, taxonomy_version'
        )
    ))
    FROM gold_validation_project_metrics AS project
    CROSS JOIN gold_validation_population_metrics AS population
    CROSS JOIN gold_validation_exposure_metrics AS exposure
    CROSS JOIN gold_validation_dimension_metrics AS dims
    CROSS JOIN gold_validation_reconciliation_summary AS reconciliation
    CROSS JOIN gold_validation_context AS context
)
SELECT
    context.validation_run_id,
    COALESCE(
        CASE checks.table_name
            WHEN 'fact_project_snapshot' THEN context.project_fact_run_id
            WHEN 'fact_region_population' THEN context.population_fact_run_id
            WHEN 'fact_region_flood_exposure' THEN context.exposure_fact_run_id
            WHEN 'dim_region' THEN context.dim_region_run_id
            WHEN 'dim_region_boundary' THEN context.dim_region_boundary_run_id
            WHEN 'dim_flood_susceptibility' THEN context.dim_flood_run_id
            WHEN 'dim_project_status' THEN context.dim_status_run_id
            WHEN 'dim_project' THEN context.dim_project_run_id
        END,
        context.validation_run_id
    ) AS pipeline_run_id,
    'gold' AS layer,
    checks.table_name,
    CASE checks.table_name
        WHEN 'fact_project_snapshot' THEN context.project_snapshot_id
        WHEN 'fact_region_flood_exposure' THEN context.mgb_source_snapshot_id
    END AS source_snapshot_id,
    validation_rule_version AS rule_version,
    checks.attribute AS data_quality_attribute,
    checks.check_name AS data_quality_check,
    checks.failed_rows,
    checks.total_rows,
    CASE WHEN checks.total_rows > 0
        THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action,
    checks.details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks
CROSS JOIN gold_validation_context AS context;

SELECT
    table_name,
    data_quality_attribute,
    data_quality_check,
    status,
    failed_rows,
    total_rows
FROM gold_validation_checks
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;

## 7. Save the evidence before the gate

**What:** `MERGE` the checks into `04-validation.gold_dq_results`, then show them, failures first.

**Why:** the saved rows stay reviewable even when the gate stops the run.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.gold_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Gold validation evidence. One row per check per validation run.';

MERGE INTO `04-validation`.gold_dq_results AS target
USING gold_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT
    table_name,
    status,
    action,
    failed_rows,
    total_rows,
    percentage,
    data_quality_check,
    details
FROM `04-validation`.gold_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM gold_validation_context)
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END, table_name, data_quality_check;

## 8. STOP gate

The run stops here if any STOP check failed.

In [0]:
%sql
SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Gold checks failed or no evidence was saved'
    )
)
FROM `04-validation`.gold_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM gold_validation_context);

## Summary

This notebook checks the Gold facts and dimensions for grain, keys, valid values, lineage and totals against Silver. Known data limits stay visible as FLAG rows with counts. It needs a Databricks run before any result can be claimed.